In [1]:
from xcube.core.store import new_data_store
import datetime

In [2]:
fsd = new_data_store("file", root="output")

In [3]:
fsd.list_data_ids()

['irrigation_input_final.zarr',
 'irrigation_input.zarr',
 'calibratedv1.zarr',
 'soil_moisture_filled.zarr']

In [4]:
irr_input = fsd.open_data("irrigation_input_final.zarr")

In [6]:
irr_input

<xarray.Dataset> Size: 548MB
Dimensions:        (time: 31, lat: 448, lon: 896)
Coordinates:
  * lat            (lat) float64 4kB 44.0 43.99 43.98 43.97 ... 40.02 40.01 40.0
  * lon            (lon) float64 7kB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996
    number         int64 8B ...
    spatial_ref    int64 8B ...
  * time           (time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
Data variables:
    SWI            (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    crs            (time) |S1 31B dask.array<chunksize=(1,), meta=np.ndarray>
    lc_binary      (lat, lon) uint8 401kB dask.array<chunksize=(448, 896), meta=np.ndarray>
    pev            (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    sf             (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    sm_filled      (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    sm_normalized  (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    ssm            (time, lat, lon) float64 100MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    ssm_noise      (time, lat, lon) float64 100MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    stl1           (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
    tp             (time, lat, lon) float32 50MB dask.array<chunksize=(1, 448, 896), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...

In [21]:
import numpy as np
from matplotlib import pyplot as plt
from netCDF4 import Dataset
import datetime as dt
import pandas as pd
from scipy.optimize import minimize
import h5py


In [23]:

#CALIBRATION
def sm_inversion(sm, et, a, b, z, RF, thr=None): 
    """Evotranspiration and Soil moisture to irrigation"""
    # sm - soil moisture
    # et - evotranspiration
    # a/b - drainage parameter
    # z - soil water capacity
    # f/RF - Adjusting factor
    p_sim = z * (sm[1:] - sm[:-1]) + ((a * sm[1:]**b + a * sm[:-1]**b) / 2.) +  ((RF * sm[1:] * et[1:] + RF * sm[:-1] * et[:-1]) / 2.)

    p_sim[abs(np.diff(sm))<=0.001]=0.0
    p_sim[p_sim < 1.0]=0.0   # 2mm/day for NN=4 -> 0.5

    return np.clip(p_sim, 0, thr)

def calib_sm_inversion(sm,  p_obs, et, NN,   x0=None, bounds=None, options=None, method='TNC'): 


    if x0 is None:
        x0 = np.array([20., 5., 80,1.])

    if bounds is None:
        bounds = ((0,200), (0.01, 50), (1, 800), (0.1,1.4))

    if options is None:
        options = {'ftol': 1e-8, 'maxiter': 4000, 'disp': False}

#    if options is None:
#        options = {'ftol': 1e-8, 'maxiter': 3000, 'disp': False}


    result = minimize(cost_fun, x0, args=(sm,  p_obs, et, NN),method=method, bounds=bounds, options=options)

    a, b, z,RF = result.x

    return a, b, z, RF


def cost_fun(x0, sm,  p_obs, et, NN):

    # The following args are 1D time-series
    p_sim = sm_inversion(sm, et, x0[0], x0[1], x0[2],x0[3])
    p_obs = p_obs[:-1]
    p_sim[np.isnan(p_obs)] = np.nan
    p_sim1 = np.add.reduceat(p_sim[np.isfinite(p_sim)], np.arange(0, len(p_sim[np.isfinite(p_sim)]), NN))
    p_obs1 = np.add.reduceat(p_obs[np.isfinite(p_sim)], np.arange(0, len(p_obs[np.isfinite(p_sim)]), NN))
    rmsd = np.nanmean((p_obs1 - p_sim1)**2)**0.5

    return rmsd


The calibration function expects a 1D array of single pixel time-series for soil moisture, evotranspiration and rainfall.

Define irrigation season
- may to september (will be refined later)
- dataset that will be published will contain those months and 0 for others - aggregated at 1 or 2 weeks
- from jan-may and sep-dec per year, we take all days for calibration.
- then inside the range may-sep, we also calibrate where rainfall < 1mm pixel-wise
- then based on the calibrated values, we calculate the IWU for may-sep

- create a new empty dataset with same dims lat, lon
- 

In [24]:
def calib_wrapper(sm_ts, p_obs_ts, et_ts, NN):
    if np.isnan(np.nanmean(sm_ts)):
        return np.array([np.nan, np.nan, np.nan, np.nan])

    a, b, z, RF = calib_sm_inversion(sm_ts, p_obs_ts, et_ts, NN)
    return np.array([a, b, z, RF])

In [9]:
from xcube.core.chunk import chunk_dataset

In [29]:
ds = chunk_dataset(irr_input, chunk_sizes={"time":-1, "lat":128, "lon":128}, format_name="zarr")
ds

<xarray.Dataset> Size: 548MB
Dimensions:        (time: 31, lat: 448, lon: 896)
Coordinates:
  * lat            (lat) float64 4kB 44.0 43.99 43.98 43.97 ... 40.02 40.01 40.0
  * lon            (lon) float64 7kB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996
    number         int64 8B ...
    spatial_ref    int64 8B ...
  * time           (time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
Data variables:
    SWI            (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    crs            (time) |S1 31B dask.array<chunksize=(31,), meta=np.ndarray>
    lc_binary      (lat, lon) uint8 401kB dask.array<chunksize=(128, 128), meta=np.ndarray>
    pev            (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sf             (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sm_filled      (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sm_normalized  (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    ssm            (time, lat, lon) float64 100MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    ssm_noise      (time, lat, lon) float64 100MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    stl1           (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    tp             (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...

In [30]:
import xarray as xr
import numpy as np
import pandas as pd
result = xr.apply_ufunc(
    calib_wrapper,
    ds["SWI"],      
    ds["tp"],    
    ds["pev"],       
    7,           
    input_core_dims=[["time"], ["time"], ["time"], []],  
    output_core_dims=[["params"]],  
    vectorize=True,                 
    dask="parallelized",           
    output_dtypes=[float],
    output_sizes={"params": 4}
)

/tmp/ipykernel_24549/2707992517.py:4: FutureWarning: ``output_sizes`` should be given in the ``dask_gufunc_kwargs`` parameter. It will be removed as direct parameter in a future version.
  result = xr.apply_ufunc(


In [26]:
result

<xarray.DataArray (lat: 448, lon: 896, params: 4)> Size: 13MB
dask.array<transpose, shape=(448, 896, 4), dtype=float64, chunksize=(448, 896, 4), chunktype=numpy.ndarray>
Coordinates:
  * lat          (lat) float64 4kB 44.0 43.99 43.98 43.97 ... 40.02 40.01 40.0
  * lon          (lon) float64 7kB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996
    number       int64 8B 0
    spatial_ref  int64 8B 0
Dimensions without coordinates: params

In [39]:
client.shutdown()

In [40]:
from dask.distributed import Client, LocalCluster
import xarray as xr

cluster = LocalCluster(n_workers=12, threads_per_worker=1)
client = Client(cluster)
print(client)

/home/yogesh/Projects/BC/irrigation-processor/.pixi/envs/default/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 34623 instead
  warnings.warn(


<Client: 'tcp://127.0.0.1:46699' processes=12 threads=12, memory=30.12 GiB>


2025-08-12 16:38:35,057 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle fb7dd55389d9aa33af896f436939829c initialized by task ('rechunk-merge-rechunk-transfer-468c64c0773227f769df33a5a8ced0ab', 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:37295
2025-08-12 16:38:35,363 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 70861051ad0987688b2680ca37837678 initialized by task ('rechunk-merge-rechunk-transfer-61bcc705cb25ee354e65ddb23eafce57', 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:46745
2025-08-12 16:38:36,734 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle db47141a191a144addfde9ec57a92a5f initialized by task ('rechunk-merge-rechunk-transfer-d7b1a65646b5e83b13b45ee8bfb47fd6', 0, 0, 0, 0, 0, 0) executed on worker tcp://127.0.0.1:39637
2025-08-12 16:38:37,052 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 70861051ad0987688b2680ca37837678 deactivated due to stimulus 'task-finished-1755009517.0347192'
2025-08-12 16:38:37,217 - di

In [41]:
%%time
ds_calib = result.compute()

/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/1440283287.py:3: RuntimeWarning: Mean of empty slice
/tmp/ipykernel_24549/2083791450.py:32: OptimizeWarning: Unknown solver options: maxiter
/tmp/ipykernel_24549/1440283

CPU times: user 21min 30s, sys: 6min 47s, total: 28min 17s
Wall time: 38min 29s


In [42]:
ds_calib

<xarray.DataArray (lat: 448, lon: 896, params: 4)> Size: 13MB
array([[[nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        ...,
        [20.,  5., 80.,  1.],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan]],

       [[nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        ...,
        [20.,  5., 80.,  1.],
        [20.,  5., 80.,  1.],
        [nan, nan, nan, nan]],

       [[nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        ...,
...
        ...,
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan]],

       [[20.,  5., 80.,  1.],
        [20.,  5., 80.,  1.],
        [20.,  5., 80.,  1.],
        ...,
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan]],

       [[20.,  5., 80.,  1.],
        [20.,  5., 80.,  1.],
        [20.,  5., 80.,  1.],
        ...,
        [nan, nan, nan, nan],
        [nan, nan, nan, nan],
        [nan, nan, nan, nan]]], shape=(448, 896, 4))
Coordinates:
  * lat          (lat) float64 4kB 44.0 43.99 43.98 43.97 ... 40.02 40.01 40.0
  * lon          (lon) float64 7kB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996
    number       int64 8B 0
    spatial_ref  int64 8B 0
Dimensions without coordinates: params

In [43]:
ds["calibration"] = ds_calib

In [47]:
fsd.write_data(ds, "calibratedv1.zarr")

'calibratedv1.zarr'

In [48]:
calibrated = fsd.open_data("calibratedv1.zarr")

In [49]:
calibrated

<xarray.Dataset> Size: 561MB
Dimensions:        (time: 31, lat: 448, lon: 896, params: 4)
Coordinates:
  * lat            (lat) float64 4kB 44.0 43.99 43.98 43.97 ... 40.02 40.01 40.0
  * lon            (lon) float64 7kB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996
    number         int64 8B ...
    spatial_ref    int64 8B ...
  * time           (time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
Dimensions without coordinates: params
Data variables:
    SWI            (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    calibration    (lat, lon, params) float64 13MB dask.array<chunksize=(112, 448, 2), meta=np.ndarray>
    crs            (time) |S1 31B dask.array<chunksize=(31,), meta=np.ndarray>
    lc_binary      (lat, lon) uint8 401kB dask.array<chunksize=(128, 128), meta=np.ndarray>
    pev            (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sf             (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sm_filled      (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    sm_normalized  (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    ssm            (time, lat, lon) float64 100MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    ssm_noise      (time, lat, lon) float64 100MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    stl1           (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
    tp             (time, lat, lon) float32 50MB dask.array<chunksize=(31, 128, 128), meta=np.ndarray>
Attributes: (12/26)
    Conventions:               CF-1.6
    archive_facility:          VITO
    copyright:                 Copernicus Service information 2020
    geospatial_lat_max:        72.0
    geospatial_lat_min:        35.0
    geospatial_lon_max:        50.0
    ...                        ...
    region_name:               CEURO
    sensor:                    CSAR
    source:                    Derived from EO radar observations
    time_coverage_end:         2020-01-01T23:59:59Z
    time_coverage_start:       2020-01-01T00:00:00Z
    title:                     Daily Surface Soil Moisture 1km: CEURO 2020-01...

In [54]:
calibrated.calibration.values.shape

(448, 896, 4)

In [55]:
arr_reshaped = calibrated.calibration.values.reshape(-1, 4)

In [56]:
unique_param_sets = np.unique(arr_reshaped, axis=0)

print(f"Number of unique parameter sets: {len(unique_param_sets)}")
print(unique_param_sets)

Number of unique parameter sets: 393219
[[0.         0.01       1.         0.1       ]
 [0.         0.01       1.         0.10434497]
 [0.         0.01       1.         0.11037029]
 ...
 [       nan        nan        nan        nan]
 [       nan        nan        nan        nan]
 [       nan        nan        nan        nan]]


In [59]:
unique_param_sets_no_nan = unique_param_sets[~np.isnan(unique_param_sets).any(axis=1)]
print(f"Unique parameter sets without NaNs: {len(unique_param_sets_no_nan)}")
print(unique_param_sets_no_nan)

Unique parameter sets without NaNs: 147145
[[0.00000000e+00 1.00000000e-02 1.00000000e+00 1.00000000e-01]
 [0.00000000e+00 1.00000000e-02 1.00000000e+00 1.04344967e-01]
 [0.00000000e+00 1.00000000e-02 1.00000000e+00 1.10370288e-01]
 ...
 [2.00000000e+02 5.00000000e+01 2.73162137e+01 1.35433848e+00]
 [2.00000000e+02 5.00000000e+01 2.77344981e+01 9.41001695e-01]
 [2.00000000e+02 5.00000000e+01 3.71946515e+01 1.11954166e+00]]


# Reading ends here. The following are not used in the calculations. Please ignore

In [28]:
lat_shape = len(swi["lat"])
lon_shape = len(swi["lon"])
lat_shape, lon_shape

(448, 896)

In [31]:
swi.isel(lat=33, lon=456).values

array([0.57      , 0.325     , 0.44326726, 0.4776992 , 0.49126017,
       0.496614  , 0.4980161 , 0.4562203 , 0.61605537, 0.665869  ,
       0.6506696 , 0.59657484, 0.5190365 , 0.45636627, 0.4873345 ,
       0.51397353, 0.5379936 , 0.56042844, 0.58190334, 0.4808122 ,
       0.48442748, 0.48662016, 0.48795006, 0.48875666, 0.4892459 ,
       0.4521629 , 0.48278946,        nan,        nan,        nan,
              nan], dtype=float32)

In [35]:
datex = [datetime.datetime(2020,1,1) + datetime.timedelta(days = i) for i in range(31)]
datex

[datetime.datetime(2020, 1, 1, 0, 0),
 datetime.datetime(2020, 1, 2, 0, 0),
 datetime.datetime(2020, 1, 3, 0, 0),
 datetime.datetime(2020, 1, 4, 0, 0),
 datetime.datetime(2020, 1, 5, 0, 0),
 datetime.datetime(2020, 1, 6, 0, 0),
 datetime.datetime(2020, 1, 7, 0, 0),
 datetime.datetime(2020, 1, 8, 0, 0),
 datetime.datetime(2020, 1, 9, 0, 0),
 datetime.datetime(2020, 1, 10, 0, 0),
 datetime.datetime(2020, 1, 11, 0, 0),
 datetime.datetime(2020, 1, 12, 0, 0),
 datetime.datetime(2020, 1, 13, 0, 0),
 datetime.datetime(2020, 1, 14, 0, 0),
 datetime.datetime(2020, 1, 15, 0, 0),
 datetime.datetime(2020, 1, 16, 0, 0),
 datetime.datetime(2020, 1, 17, 0, 0),
 datetime.datetime(2020, 1, 18, 0, 0),
 datetime.datetime(2020, 1, 19, 0, 0),
 datetime.datetime(2020, 1, 20, 0, 0),
 datetime.datetime(2020, 1, 21, 0, 0),
 datetime.datetime(2020, 1, 22, 0, 0),
 datetime.datetime(2020, 1, 23, 0, 0),
 datetime.datetime(2020, 1, 24, 0, 0),
 datetime.datetime(2020, 1, 25, 0, 0),
 datetime.datetime(2020, 1, 26, 0,

In [45]:
tp.shape[0]

31

In [40]:
import numpy as np
import pandas as pd

In [46]:
rain_flat = np.empty((tp.shape[0], tp.shape[1]*tp.shape[2]))
# for i in range(tp.shape[0]):
rain_flat = tp.stack(pixel_number=("lat", "lon"))

In [47]:
rain_flat

<xarray.DataArray 'tp' (time: 31, pixel_number: 401408)> Size: 50MB
dask.array<reshape, shape=(31, 401408), dtype=float32, chunksize=(1, 401408), chunktype=numpy.ndarray>
Coordinates:
    number        int64 8B ...
    spatial_ref   int64 8B ...
  * time          (time) datetime64[ns] 248B 2020-01-01 ... 2020-01-31
  * pixel_number  (pixel_number) object 3MB MultiIndex
  * lat           (pixel_number) float64 3MB 44.0 44.0 44.0 ... 40.0 40.0 40.0
  * lon           (pixel_number) float64 3MB -4.996 -4.987 ... 2.987 2.996
Attributes: (12/33)
    GRIB_NV:                                  0
    GRIB_Nx:                                  80
    GRIB_Ny:                                  40
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           tp
    GRIB_dataType:                            fc
    ...                                       ...
    GRIB_units:                               mm
    GRIB_uvRelativeToGrid:                    0
    grid_mapping:                             spatial_ref
    long_name:                                Total precipitation (millimeters)
    standard_name:                            unknown
    units:                                    mm

In [48]:
RAIN_dataframe = pd.DataFrame(rain_flat[:-1,:], index = datex[:-1])

mask1 = (RAIN_dataframe.index.month > 4) 
mask2 = (RAIN_dataframe.index.month < 10)
mask3 = (RAIN_dataframe[mask1 & mask2]) < 1.0

RAIN_dataframe[mask3] = np.nan

In [49]:
RAIN_dataframe

,0,1,2,3,4,5,6,7,8,9,...,401398,401399,401400,401401,401402,401403,401404,401405,401406,401407
2020-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-07,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-08,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-01-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print ('FLATTEN ARRAYS')
print (SWI_masked_flat.shape)
print (rainfall_masked_flat.shape)
print (petdaily_masked_flat.shape)


print ('CALIBRATION OF a, b,  Z, and RF')

a = np.empty((sm.shape[1]*sm.shape[2]))
b = np.empty((sm.shape[1]*sm.shape[2]))
z = np.empty((sm.shape[1]*sm.shape[2]))
f = np.empty((sm.shape[1]*sm.shape[2]))

for i in range(sm.shape[1]*sm.shape[2]):

    if np.isnan(np.nanmean(SWI_masked_flat[:,i])):
        
        a[i] = np.nan
        b[i] = np.nan
        z[i] = np.nan
        f[i] = np.nan
    else:
        print ('VALID POINT', i)
        H = calib_sm_inversion(SWI_masked_flat[:,i],rainfall_masked_flat[:,i],petdaily_masked_flat[:,i], 7) 
        a[i] = H[0]
        b[i] = H[1]
        z[i] = H[2]
        f[i] = H[3]
        

- chunk the data in time dimension and lat, lon 1
- use map_blocks to do the calibration
- 

In [ ]:
for i in range(lat_shape):
    for j in range(lon_shape):
        swi_input = swi.isel(lat=i, lon=j)
        

In [9]:
swi_stacked = swi.stack(pixel=("lat", "lon"))
swi_stacked

<xarray.DataArray 'SWI' (time: 31, pixel: 401408)> Size: 50MB
dask.array<reshape, shape=(31, 401408), dtype=float32, chunksize=(1, 401408), chunktype=numpy.ndarray>
Coordinates:
    number       int64 8B ...
    spatial_ref  int64 8B ...
  * time         (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
  * pixel        (pixel) object 3MB MultiIndex
  * lat          (pixel) float64 3MB 44.0 44.0 44.0 44.0 ... 40.0 40.0 40.0 40.0
  * lon          (pixel) float64 3MB -4.996 -4.987 -4.978 ... 2.978 2.987 2.996

In [19]:
swi_stacked.isel(pixel=78651).values

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan], dtype=float32)

In [23]:
swi_stacked.dropna(dim="pixel")

<xarray.DataArray 'SWI' (time: 31, pixel: 0)> Size: 0B
dask.array<getitem, shape=(31, 0), dtype=float32, chunksize=(1, 0), chunktype=numpy.ndarray>
Coordinates:
    number       int64 8B ...
    spatial_ref  int64 8B ...
  * time         (time) datetime64[ns] 248B 2020-01-01 2020-01-02 ... 2020-01-31
  * pixel        (pixel) object 0B MultiIndex
  * lat          (pixel) float64 0B 
  * lon          (pixel) float64 0B